In [1]:
# Grand Canonical Monte Carlo (Ag nanoparticle)
#
# This notebook is split into short cells so each simulation step
# is easier to follow: imports, geometry, MC region, moves,
# thermodynamic setup, ensemble construction, and run.

In [ ]:
from ase.cluster import Octahedron
from mace.calculators import mace_mp

from mcpy.moves import DeletionMove, InsertionMove
from mcpy.moves.move_selector import MoveSelector
from mcpy.ensembles.grand_canonical_ensemble import GrandCanonicalEnsemble
from mcpy.cell import SphericalCell

## 1) Build the starting nanoparticle

`Octahedron('Ag', 6, 1)` creates an Ag cluster that will be the initial configuration for the simulation.

In [ ]:
atoms = Octahedron('Ag', 6, 1)
atoms

## 2) Define the simulation cell for insertion/deletion

`SphericalCell` wraps the particle in a spherical region (with vacuum padding) and controls where Monte Carlo insertion/deletion attempts are sampled.

In [ ]:
scell = SphericalCell(
    atoms,
    vacuum=3,
    species_radii={'Ag': 2.947, 'O': 0},
    mc_sample_points=100_000,
)
scell

## 3) Choose the energy calculator

The Monte Carlo acceptance test needs energies. Here we use `mace_mp` (a pretrained MACE model) on GPU (`device='cuda'`).

In [ ]:
calculator = mace_mp(device='cuda')

## 4) Configure Monte Carlo moves

- We only sample oxygen (`species = ['O']`).
- `DeletionMove` tries removing O atoms.
- `InsertionMove` tries adding O atoms.
- `MoveSelector` picks between them using the provided weights.

In [ ]:
species = ['O']

move_list = [
    [1, 1],
    [
        DeletionMove(scell, species=['O'], seed=43215423143),
        InsertionMove(scell, species=['O'], min_insert=0.5, seed=3675437856),
    ],
]

move_selector = MoveSelector(*move_list)
move_selector

## 5) Set thermodynamic conditions

`mu` values are chemical potentials (in metal units here), and `T` is temperature in Kelvin.
The `delta_mu_O` offset is used to shift oxygen potential for this run.

In [ ]:
mus = {'Ag': -2.99, 'O': -4.91}
delta_mu_O = -0.5
mus['O'] += delta_mu_O
T = 500

mus, T

## 6) Build the GCMC ensemble object

This assembles geometry, allowed cell region, calculator, species list, move selector, and thermodynamic state into one simulation object.

In [ ]:
gcmc = GrandCanonicalEnsemble(
    atoms=atoms,
    cells=[scell],
    calculator=calculator,
    mu=mus,
    units_type='metal',
    species=species,
    temperature=T,
    move_selector=move_selector,
)

gcmc

## 7) Run the simulation

This is the original long production run (`1_000_000` MC steps). For quick testing, temporarily reduce `n_steps` to something like `1_000` or `10_000`.

In [ ]:
n_steps = 1_000_000
gcmc.run(n_steps)